# Simple API Image Downloader for Dataset Creation

## Project Description
I made this notebook to automate the boring and time-consuming part of deep learning projects: collecting data. The code has a clean and flexible structure to search, download, and sort images. By just updating the API endpoint and the keys, you can easily change it to download images from different sites like **Pexels**, **Pixabay**, or **Unsplash**.

## Key Features
- **Easy API Swap:** The code is flexible. You can switch between Pexels, Pixabay, or Unsplash APIs simply by changing the request URL and credentials.
- **Auto-Folders:** It automatically creates folders for each class (like `data/horse` and `data/zebra`) so the images are immediately ready to train the model.
- **Looping Pages:** It uses basic loops to handle API page limits, making sure we get the exact number of balanced images we want.
- **Colab & Drive Backup:** Works perfectly with Google Colab to mount Google Drive and back up the images automatically, so nothing gets lost.


## 0. Configuration

All file paths used in this notebook are derived from the variables
below — matching the convention used in `starter.ipynb`. Only
`API_KEY` and `DRIVE_DIR` need to be changed if you re-run this on your
own Colab/Drive setup.


In [ ]:
import os

BASE_DIR = "/content"                 # local Colab runtime working dir
DRIVE_DIR = "/content/drive/MyDrive"  # persistent storage (Google Drive)

DATA_DIR = f"{BASE_DIR}/data"            # downloaded images (horse/, zebra/)
OUTPUT_DIR = f"{DRIVE_DIR}/pexels_data"  # Drive backup destination

API_KEY = "-----------"  # Pexels API key — get one for free at https://www.pexels.com/api/


In [ ]:
!pip install requests --quiet


In [ ]:
import shutil
shutil.rmtree(DATA_DIR, ignore_errors=True)
print("Clean slate — old data folder removed.")


In [ ]:
import requests
import os
import time

headers = {"Authorization": API_KEY}


def download_pexels_images(query, folder, count=30):
    os.makedirs(folder, exist_ok=True)
    downloaded = 0
    page = 1
    while downloaded < count:
        url = f"https://api.pexels.com/v1/search?query={query}&per_page=80&page={page}"
        response = requests.get(url, headers=headers)
        data = response.json()
        if 'photos' not in data or len(data['photos']) == 0:
            break
        for photo in data['photos']:
            if downloaded >= count:
                break
            filename = f"{folder}/pexels_{photo['id']}.jpg"
            if os.path.exists(filename):
                continue
            img_data = requests.get(photo['src']['large']).content
            with open(filename, 'wb') as f:
                f.write(img_data)
            downloaded += 1
        page += 1
        time.sleep(0.5)
    print(f"'{query}': {downloaded} files downloaded -> {folder}")


In [ ]:
horse_queries = [
    ("horse portrait", 20),
    ("horse in field", 20),
    ("arabian horse", 15),
    ("friesian horse", 15),
    ("appaloosa horse", 15),
    ("clydesdale horse", 15),
    ("wild mustang", 15),
    ("white horse", 15),
    ("black horse", 15),
    ("grey horse", 15),
    ("horse close up face", 15),
    ("horse full body side view", 15),
    ("horse running distant", 15),
    ("horse aerial view field", 15),
    ("horse lying down", 15),
    ("foal", 15),
    ("baby horse", 15),
    ("young horse", 15),
]

zebra_queries = [
    ("zebra portrait", 20),
    ("zebra in savanna", 20),
    ("plains zebra", 15),
    ("grevy zebra", 15),
    ("mountain zebra", 15),
    ("zebra close up face", 15),
    ("zebra full body side view", 15),
    ("zebra running distant", 15),
    ("zebra aerial view savanna", 15),
    ("zebra herd", 15),
    ("zebra lying down", 15),
    ("baby zebra", 15),
    ("zebra foal", 15),
    ("young zebra", 15),
    ("zebra drinking water", 15),
    ("zebra grazing", 15),
    ("zebra walking", 15),
    ("zebra resting", 15),
]

for query, count in horse_queries:
    download_pexels_images(query, f"{DATA_DIR}/horse", count=count)

for query, count in zebra_queries:
    download_pexels_images(query, f"{DATA_DIR}/zebra", count=count)

print(f"\nhorse: {len(os.listdir(f'{DATA_DIR}/horse'))}")
print(f"zebra: {len(os.listdir(f'{DATA_DIR}/zebra'))}")
